In [52]:
import requests
import pandas as pd
import numpy as np
import feedparser
import os
from openai import OpenAI
import gradio as gr
import json
import time

In [53]:
model= "gpt-4o-mini"
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))


In [54]:
class FundamentalAgent:

    def fetch_news(self, asset):

        url = f"https://news.google.com/rss/search?q={asset}+stock"
        feed = feedparser.parse(url)

        headlines = []

        for entry in feed.entries[:5]:
            headlines.append(entry.title)

        return headlines


    def analyze(self, asset):

        headlines = self.fetch_news(asset)

        prompt = f"""
            You are a financial fundamental analyst.
            Asset: {asset}
            News Headlines:
            {headlines}
            Based on the sentiment of the news decide:
            BUY, SELL, or HOLD.
            Explain briefly why.
        """

        response = client.chat.completions.create(
            model=model,
            messages=[{"role":"user","content":prompt}]
        )

        return response.choices[0].message.content


In [68]:

class TechnicalAgent:

    def _yahoo_symbol(self, asset):
        """Normalize asset symbol for Yahoo Finance (e.g. BTC_USD -> BTC-USD)."""
        return asset.replace("_", "-").upper()

    def fetch_price(self, asset):

        symbol = self._yahoo_symbol(asset)
        
        url = f"https://query1.finance.yahoo.com/v8/finance/chart/{symbol}?range=3mo&interval=1d"
        resp = requests.get(url)

        if not resp.ok:
            raise ValueError(f"Yahoo Finance error {resp.status_code} for {asset}: {resp.text[:200]}")

        try:
            data = resp.json()
        except requests.JSONDecodeError:
            raise ValueError(f"Yahoo Finance returned non-JSON for {asset}. Check symbol (e.g. use BTC-USD or TSLA).")

        if not data.get("chart", {}).get("result"):
            raise ValueError(f"No chart data for {asset}. Symbol may be invalid (e.g. use BTC-USD, AAPL, TSLA).")

        prices = data["chart"]["result"][0]["indicators"]["quote"][0]["close"]

        df = pd.DataFrame({"close": prices})
        df = df.dropna()

        return df


    def analyze(self, asset):

        df = self.fetch_price(asset)

        df["MA20"] = df["close"].rolling(20).mean()
        df["MA50"] = df["close"].rolling(50).mean()

        latest = df.iloc[-1]

        prompt = f"""
            You are a technical analyst.
            Latest Data:
            Price: {latest['close']}
            MA20: {latest['MA20']}
            MA50: {latest['MA50']}
            If MA20 > MA50 → bullish.
            If MA20 < MA50 → bearish.
            Return BUY, SELL or HOLD with a short explanation.
        """

        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role":"user","content":prompt}]
        )

        return response.choices[0].message.content

In [69]:
class DecisionAgent:

    def decide(self, fundamental, technical):

        prompt = f"""
            You are a portfolio decision agent.

            Fundamental Analysis:
            {fundamental}

            Technical Analysis:
            {technical}

            Decide the FINAL action:

            STRONG BUY
            BUY
            HOLD
            SELL
            STRONG SELL

            Explain briefly.
        """

        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role":"user","content":prompt}]
        )

        return response.choices[0].message.content


In [ ]:
fundamental_agent = FundamentalAgent()
technical_agent = TechnicalAgent()
decision_agent = DecisionAgent()


def analyze(asset):

    fundamental = fundamental_agent.analyze(asset)

    #Crucial because of yahoo finance rate limit
    time.sleep(2)
    
    technical = technical_agent.analyze(asset)

    final = decision_agent.decide(fundamental, technical)

    return f"""
        FINAL DECISION
        {final}

        --------------------

        FUNDAMENTAL ANALYSIS
        {fundamental}

        --------------------

        TECHNICAL ANALYSIS
        {technical}
    """

analyze_function = {
    "name": "analyze",
    "description": "Analyze a stock or cryptocurrency",
    "parameters": {
        "type": "object",
        "properties": {
            "asset": {"type": "string", "description": "The asset to analyze in yahoo finance format e.g BTC-USD, AAPL, TSLA, etc"}
        },
        "required": ["asset"]
    }
}

tools = [ {"type": "function", "function": analyze_function} ]

In [75]:
def handle_tool_calls(message):
    responses = []
    tool_call = message.tool_calls[0]
    for tool_call in message.tool_calls:
        tool_name = tool_call.function.name
        if tool_name == "analyze":
            print(tool_call.function)
            arguments = json.loads(tool_call.function.arguments)
            responses.extend({
                "role": "tool",
                "content": analyze(**arguments),
                "tool_call_id": tool_call.id
            })
        else:
            responses.extend({
                "role": "tool",
                "content": "Tool call not recognized",
                "tool_call_id": tool_call.id
            })
    return responses

In [76]:
system_prompt = f"""
You are a financial analysis agent.
You can analyze stocks, cryptocurrencies, and other assets.
using only the tools provided, don't use your own knowledge
"""
def chat(message, history):
    system = {"role": "system", "content": system_prompt}
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [system] + history + [{"role": "user", "content": message}]
    response = client.chat.completions.create(model=model, messages=messages, tools=tools)
    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = client.chat.completions.create(model=model, messages=messages, tools=tools)

    return response.choices[0].message.content
    

In [77]:
technical_agent.analyze("BTC-USD")


ValueError: Yahoo Finance error 429 for BTC-USD: Edge: Too Many Requests

In [78]:
ui = gr.ChatInterface(
    fn=chat,
    type="messages",
    title="Financial Analysis Agent",
    description="Ask the agent about financial analysis of a stock or cryptocurrency.",
)

ui.launch()

* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


Function(arguments='{"asset":"BTC-USD"}', name='analyze')


Traceback (most recent call last):
  File "/Users/taiwoifedayo/Tutorials/AndelaBootcamp/llm_engineering/.venv/lib/python3.12/site-packages/gradio/queueing.py", line 759, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/taiwoifedayo/Tutorials/AndelaBootcamp/llm_engineering/.venv/lib/python3.12/site-packages/gradio/route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/taiwoifedayo/Tutorials/AndelaBootcamp/llm_engineering/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 2116, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/taiwoifedayo/Tutorials/AndelaBootcamp/llm_engineering/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 1621, in call_function
    prediction = await fn(*processed_input)
                 ^^^^^^^^^^^^^^^